# A/B: RelBench's in-memory path against ours (Plan 03)

Do the changes keep the accuracy, and what do they cost in time? This trains the same model the same way on a dataset that fits in memory both ways, and compares:

| Arm | `--features-on-disk` | How the graph is built |
|---|---|---|
| `A_relbench` | `none` | RelBench's `get_db()` + `make_pkey_fkey_graph`: all tables and features in memory |
| `B_ours` | `text` | tables read one at a time, text vectors on disk, memory-mapped |

**Both arms use the same settings:**
- the model: RelGraphSAGE (our Base model);
- RelBench's settings: 10 epochs, lr 0.005, batch 512, hidden 128, 2 layers, 128 neighbours;
- GloVe text;
- the same seeds, on the same device.

**Each (arm, seed) is a fresh process** (`experiments/benchmarks/ab_compare.py`).
- Each arm's first seed builds its graph from the raw tables; later seeds load it from that arm's own cache.
- Training starts from the same random state in both arms. The two paths draw different amounts of random numbers while loading, so the runner reseeds after loading.

Measured, per run:
- validation and test ROC-AUC;
- time to load the graph (first seed: built from the raw tables);
- time per epoch;
- time of one prediction pass;
- peak RAM, ordinary and file-backed, and peak GPU memory.

**Run time.** rel-arxiv's task is large: about 1,350 batches per epoch (1,044 training, 305 validation). The defaults are a quick check, 2 epochs × seeds 0 and 1 × 2 arms, roughly 30–60 minutes on a T4 by estimate. RelBench's full setting (10 epochs, 3 seeds) would take several hours. Each epoch prints a line, so the time between two lines × the number of epochs left is the real estimate.

**On Colab:**
1. *Runtime → Change runtime type → T4 GPU*. With no GPU, the notebook still runs, on CPU, and says so in a banner.
2. *Runtime → Run all*.
3. The last cell downloads `results.csv`. Upload it, and this notebook saved with its outputs (*File → Download → .ipynb*).


In [ ]:
# ---- Settings ----
PROJECT_REPO = 'https://github.com/HARSHALK2598/cs224w-structural-optimizer'
PROJECT_BRANCH = 'step/03-2-tables-one-at-a-time'
PYG_REPO = 'https://github.com/HARSHALK2598/pytorch_geometric'
PYG_BRANCH = 'contrib/structural-optimizer'
RELBENCH_COMMIT = '91c1f6420a03feb61c279aaeb194d7268435f278'  # as in scripts/setup_env.sh

DATASET = 'rel-arxiv'
TASK = 'paper-citation'        # binary classification on papers
SEEDS = [0, 1]                 # a quick check; [0, 1, 2] for the full comparison
ARMS = ['A_relbench', 'B_ours']
EPOCHS = 2                     # a quick check; None for RelBench's setting (10)
REQUIRE_GPU = False            # True: stop if there is no GPU
FRESH_CACHE = True             # rebuild each arm's graph, so the first seed measures the build
USE_DRIVE = True               # Colab only: keep results in Google Drive
DRIVE_DIR = '/content/drive/MyDrive/cs224w-structural-optimizer'


## 1. Environment

Clones both repositories and installs pinned versions on Colab; uses the current environment elsewhere. Prints the device and the pandas version: with pandas 3, missing texts stay missing before embedding, and with pandas 2 they become the text `'nan'`. Both arms see the same either way.


In [ ]:
import os, subprocess, sys, shutil
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules


def sh(cmd, cwd=None):
    """Runs a shell command, streaming its output; stops on failure."""
    print('$', cmd, flush=True)
    # PYTHONUNBUFFERED: show the sweep's lines as they happen, not at the end.
    proc = subprocess.Popen(cmd, shell=True, cwd=cwd, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True,
                            env={**os.environ, 'PYTHONUNBUFFERED': '1'})
    for line in proc.stdout:
        print(line, end='', flush=True)
    if proc.wait() != 0:
        raise RuntimeError(f'command failed ({proc.returncode}): {cmd}')


HAS_GPU = shutil.which('nvidia-smi') is not None and subprocess.run(
    ['nvidia-smi'], capture_output=True).returncode == 0
if HAS_GPU:
    print('=' * 70 + '\nRUNNING ON GPU\n' + '=' * 70, flush=True)
else:
    print('=' * 70 + '\nRUNNING ON CPU: no GPU found. Training will be much slower.\n'
          'For a GPU: Runtime -> Change runtime type -> T4 GPU, then run all again.\n'
          + '=' * 70, flush=True)
    if REQUIRE_GPU:
        raise RuntimeError('REQUIRE_GPU is set and there is no GPU')

if IN_COLAB:
    ROOT = Path('/content/cs224w-structural-optimizer')
    if not ROOT.exists():
        sh(f'git clone -q --branch {PROJECT_BRANCH} {PROJECT_REPO} {ROOT}')
    else:
        sh(f'git -C {ROOT} pull -q origin {PROJECT_BRANCH}')
    pyg = ROOT / 'pytorch_geometric'
    if not pyg.exists():
        sh(f'git clone -q --branch {PYG_BRANCH} {PYG_REPO} {pyg}')
    else:
        sh(f'git -C {pyg} pull -q origin {PYG_BRANCH}')

    build = 'cu126' if HAS_GPU else 'cpu'
    pip = f'{sys.executable} -m pip install -q'
    sh(f'{sys.executable} -m pip uninstall -q -y torchvision torchaudio')
    sh(f'{pip} torch==2.12.0 --index-url https://download.pytorch.org/whl/{build}')
    sh(f'{pip} pyg_lib==0.9.0 -f https://data.pyg.org/whl/torch-2.12.0+{build}.html')
    sh(f'{pip} -e {pyg}')
    sh(f'{pip} "relbench @ git+https://github.com/snap-stanford/relbench@{RELBENCH_COMMIT}" '
       f'pytorch-frame==0.3.0 sentence-transformers==6.2.0 pyyaml pandas')
else:
    # Run from anywhere inside the project repository (or set PROJECT_ROOT).
    ROOT = Path(os.environ.get('PROJECT_ROOT', Path.cwd())).resolve()
    while not (ROOT / 'experiments' / 'sweep.py').exists():
        if ROOT.parent == ROOT:
            raise RuntimeError('Start the notebook inside the project repository.')
        ROOT = ROOT.parent

os.chdir(ROOT)
print('project root:', ROOT, '| Colab:', IN_COLAB, '| GPU:', HAS_GPU)
sh(f'{sys.executable} -c "import torch, torch_geometric, torch_frame, relbench, pyg_lib; '
   f'print(\'torch\', torch.__version__, \'| cuda\', torch.cuda.is_available(), '
   f'torch.cuda.get_device_name(0) if torch.cuda.is_available() else \'\', '
   f'\'| pyg\', torch_geometric.__version__, \'| frame\', torch_frame.__version__, '
   f'\'| relbench\', relbench.__version__, \'| pyg_lib\', pyg_lib.__version__)"')
sh(f'{sys.executable} -c "import pandas, numpy; print(\'pandas\', pandas.__version__, \'| numpy\', numpy.__version__)"')
sh('git log --oneline -1')
sh('git -C pytorch_geometric log --oneline -1' if (ROOT / 'pytorch_geometric').exists() else 'true')

## 2. Where results go

One row per (arm, seed) is added to `ab_compare/results.csv`: in Google Drive on Colab with `USE_DRIVE = True`, in the repository otherwise.


In [ ]:
if IN_COLAB and USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT = Path(DRIVE_DIR) / 'ab_compare' / 'results.csv'
else:
    OUT = ROOT / 'experiments' / 'results' / 'ab_compare' / 'results.csv'
OUT.parent.mkdir(parents=True, exist_ok=True)
CACHE_ROOT = ROOT / 'data' / 'materialized_ab'
if FRESH_CACHE:
    shutil.rmtree(CACHE_ROOT / DATASET, ignore_errors=True)
print('results file:', OUT, '| cache:', CACHE_ROOT)
sh('free -g; df -h /; nvidia-smi --query-gpu=name,memory.total --format=csv || true')


## 3. Download the dataset

Only this dataset's folder: its tables and task labels.

In [ ]:
sh(f'{sys.executable} -c "import os; os.environ.setdefault(\'HF_HOME\', \'{ROOT}/data/hf\'); '
   f'from relbench import load_dataset; d = load_dataset(\'{DATASET}\'); print(d.dataset_dir)"')


## 4. The runs

Arm A's seeds first, then arm B's. Each run prints its per-epoch log and one result line.

In [ ]:
epochs = f' --epochs {EPOCHS}' if EPOCHS else ''
for arm in ARMS:
    for seed in SEEDS:
        print(f'\n======== {arm} seed {seed} ========', flush=True)
        try:
            sh(f'{sys.executable} experiments/benchmarks/ab_compare.py --arm {arm} --seed {seed} '
               f'--dataset {DATASET} --task {TASK}{epochs} --cache-root "{CACHE_ROOT}" --out "{OUT}"')
        except RuntimeError as error:
            print('FAILED:', error, flush=True)


## 5. Results

1. **Accuracy** per arm (mean ± std over seeds), and the difference between the arms seed by seed.
2. **Time and memory** per arm: the build (first seed), loading from cache (later seeds), per epoch, one prediction pass, and peak memory.


In [ ]:
import pandas as pd
pd.set_option('display.width', 200)
res = pd.read_csv(OUT)
res = res[(res['dataset'] == DATASET) & (res['task'] == TASK)]
print(res.to_string(index=False), '\n')

print('1. Accuracy (', res['metric'].iloc[0], ')')
acc = res.groupby('arm')[['val', 'test']].agg(['mean', 'std']).round(4)
print(acc.to_string(), '\n')
paired = res.pivot_table(index='seed', columns='arm', values=['val', 'test'])
for split in ['val', 'test']:
    if {'A_relbench', 'B_ours'} <= set(paired[split].columns):
        diff = paired[split]['B_ours'] - paired[split]['A_relbench']
        print(f'   {split}: B - A per seed:', diff.round(4).to_dict())

print('\n2. Time (seconds) and memory (GB)')
built = res[res['cache_was_built']].groupby('arm')['load_s'].mean().rename('build_s (first seed)')
cached = res[~res['cache_was_built']].groupby('arm')['load_s'].mean().rename('load_from_cache_s')
rest = res.groupby('arm')[['epoch_s_mean', 'predict_val_s', 'peak_anon_gb', 'peak_file_gb', 'gpu_peak_gb']].mean()
print(pd.concat([built, cached, rest], axis=1).round(2).to_string())
print('\ndevice:', sorted(res['device'].astype(str).unique()), sorted(res['gpu'].fillna('').astype(str).unique()),
      '| pandas', sorted(res['pandas'].astype(str).unique()))


## 6. Download the results

In [ ]:
if IN_COLAB:
    from google.colab import files
    files.download(str(OUT))
